In [ ]:
print(5)

5


In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# os.environ['OPENAI_ENDPOINT'] = os.getenv("OPENAI_ENDPOINT")

In [3]:
from agents import Agent, Runner

agent = Agent(name="Assistant", instructions="You are a helpful assistant")

result = await Runner.run_sync(agent, "Write a haiku about recursion in programming.")
print(result.final_output)

RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.

In [25]:
# !pip install langchain-openai

from langchain_openai import OpenAI

llm = OpenAI(
    temperature = 0,
    timeout = None,
    max_retries = 2,
)

llm


OpenAI(client=<openai.resources.completions.Completions object at 0x0000019EF02FEBF0>, async_client=<openai.resources.completions.AsyncCompletions object at 0x0000019EF03377F0>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [26]:
llm.invoke("hello")

'\n\n\nHello! How are you doing today?'

In [ ]:
# !pip install langchain-community

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.example.com/")

  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 36.6 MB/s  0:00:00
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 48.7 MB/s  0:00:00
Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 58.6 MB/s  0:00:00
Using cached typing_inspect-0.9.0-py3-none-any.whl (8.8 kB)
Using cached aiohappyeyeballs-2.6.1

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# !pip install bs4

docs = loader.load()

  Using cached bs4-0.0.2-py2.py3-none-any.whl.metadata (411 bytes)
Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)

   ------------- -------------------------- 1/3 [beautifulsoup4]
   ---------------------------------------- 3/3 [bs4]



In [32]:
!pip install -qU "langchain-chroma>=0.1.2"

In [33]:
# | output: false
# | echo: false
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [36]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(docs, embeddings, collection_name="example")

In [35]:
import chromadb

client = chromadb.Client()

In [37]:
vector_store

In [39]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.invoke("ming chi university of technology")

[Document(id='44f3ffa1-119f-49ba-a9ff-f68285a1169f', metadata={'source': 'https://www.example.com/', 'language': 'en', 'title': 'Example Domain'}, page_content='Example DomainExample DomainThis domain is for use in documentation examples without needing permission. Avoid use in operations.Learn more\n')]

In [5]:
## extract all links from the base link
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

visited = set()

def extract_links(url, base_domain):
    if url in visited:
        return
    
    visited.add(url)

    try:
        response = requests.get(url, timeout=5)
        soup = BeautifulSoup(response.text, "html.parser")

        for link in soup.find_all("a", href=True):
            href = link["href"]
            absolute_url = urljoin(url, href)

            # stay inside same domain
            if urlparse(absolute_url).netloc == base_domain:
                print(absolute_url)
                extract_links(absolute_url, base_domain)

    except Exception as e:
        pass


start_url = "https://www.uc.edu/"
domain = urlparse(start_url).netloc

extract_links(start_url, domain)

print(f"\nTotal links found: {len(visited)}")

https://www.uc.edu/#main
https://www.uc.edu/#main
https://www.uc.edu
https://www.uc.edu#main
https://www.uc.edu#main
https://www.uc.edu
https://www.uc.edu/connect.html
https://www.uc.edu/connect.html#main
https://www.uc.edu/connect.html#main
https://www.uc.edu
https://www.uc.edu/connect.html
https://www.uc.edu/scholarships-financial-aid.html
https://www.uc.edu/scholarships-financial-aid.html#main
https://www.uc.edu/scholarships-financial-aid.html#main
https://www.uc.edu
https://www.uc.edu/connect.html
https://www.uc.edu/scholarships-financial-aid.html
https://www.uc.edu/connect.html
https://www.uc.edu/scholarships-financial-aid.html
https://www.uc.edu/admissions.html
https://www.uc.edu/admissions.html#main
https://www.uc.edu/admissions.html#main
https://www.uc.edu
https://www.uc.edu/connect.html
https://www.uc.edu/scholarships-financial-aid.html
https://www.uc.edu/connect.html
https://www.uc.edu/scholarships-financial-aid.html
https://www.uc.edu/admissions.html
https://www.uc.edu/about

KeyboardInterrupt: 

In [8]:
with open("links.txt", "r") as f:
    links = f.readlines()

links = [link.strip() for link in links]
links

['https://www.uc.edu/news/blog/career-ready/upskilling.html',
 'https://www.uc.edu/about/financial-aid/aid/scholarships/grad-aid.html',
 'https://www.uc.edu/campus-life/food/nutrition/healthy-eating-tips.html',
 'https://www.uc.edu/about/international/visa/faculty/tn.html',
 'https://www.uc.edu/about/financial-aid/manage-aid/year-round-aid.html',
 'https://www.uc.edu/news/articles/2023/02/n21146243.html',
 'https://www.uc.edu/about/provost/colleges-and-offices/colleges.html',
 'https://www.uc.edu/news/articles/2025/05/n21331114.html',
 'https://www.uc.edu/about/financial-aid/manage-aid/loan-management/debt-management.html',
 'https://www.uc.edu/news/articles/2026/01/n21380222.html',
 'https://www.uc.edu/about/digital-accessibility/product-specific-guides/adobe.html',
 'https://www.uc.edu/about/digital-accessibility/core-concepts.html',
 'https://www.uc.edu/about/ucit/about/cybersecurity/education---awareness/events.html',
 'https://www.uc.edu/bursar/fees.html',
 'https://www.uc.edu/adm